# 🚀 AI Voice Studio - GPT-SoVITS 무료 Google Colab GPU API 서버

<a href="https://colab.research.google.com/github/ssss2513-cyber/ai-audio-studio/blob/main/GPT_SoVITS_Colab_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
### 💡 이 노트북의 역할
- 구글이 무료로 제공하는 **Nvidia T4 GPU (16GB)** 그래픽카드를 활용하여 **GPT-SoVITS 목소리 복제 API**를 구동합니다.
- **내 컴퓨터를 꺼두어도**, **외장 그래픽카드가 없는 컴퓨터나 스마트폰에서도** 자유롭게 목소리 복제를 쓸 수 있습니다.
- 원격 웹사이트([AI Voice Studio](https://voice-studio.streamlit.app/))와 완벽 호환되도록 오디오 자동 업로드 및 Cloudflare 인터넷 터널 주소를 발급합니다.

### ⚡ 3단계 간편 사용법
1. 상단 메뉴 **런타임** ➔ **모두 실행** (`Ctrl + F9`) 클릭
2. 약 2~3분 뒤 마지막 4단계 셀에서 출력되는 **`👉 복사할 API 주소: https://...trycloudflare.com/tts`** 복사
3. [AI Voice Studio 사이트](https://voice-studio.streamlit.app/)의 좌측 사이드바 **[GPT-SoVITS API 주소]**에 붙여넣기 후 사용!

In [ ]:
# [1단계] GPU 상태 확인 및 필수 시스템 패키지 설치
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg curl wget
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("\n✅ [1/4] GPU 환경 확인 및 Cloudflared, FFmpeg 도구 설치 완료!")

In [ ]:
# [2단계] GPT-SoVITS 소스코드 다운로드 및 필수 파이썬 라이브러리 설치
import os
if not os.path.exists("/content/GPT-SoVITS"):
    !git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git /content/GPT-SoVITS
%cd /content/GPT-SoVITS
!pip install -q -r requirements.txt
!pip install -q huggingface_hub fastapi uvicorn requests
print("\n✅ [2/4] GPT-SoVITS 프레임워크 및 의존성 라이브러리 설치 완료!")

In [ ]:
# [3단계] 공식 사전 학습 AI 모델 다운로드 (Google 초고속망 사용, 약 1분 소요)
from huggingface_hub import snapshot_download
import os

models_dir = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models"
os.makedirs(models_dir, exist_ok=True)

print("⏳ 사전 학습 모델 다운로드 중... 잠시만 기다려주세요...")
snapshot_download(
    repo_id="lj1995/GPT-SoVITS",
    local_dir=models_dir,
    local_dir_use_symlinks=False
)
print("\n✅ [3/4] 사전 학습 모델 다운로드 완료!")

In [ ]:
# [4단계] 원격 오디오 전송 지원 스마트 API 서버 및 Cloudflare 터널 구동
import os
import sys
import time
import subprocess
import re

if not os.path.exists('/content/GPT-SoVITS'):
    print('⚠️ [주의] 1~3단계가 아직 완료되지 않았습니다!')
    print('상단 메뉴에서 [런타임] ➔ [모두 실행] (Ctrl + F9)을 클릭해 1단계부터 차례대로 실행해주세요.')
    sys.exit(1)

%cd /content/GPT-SoVITS

# 원격 웹 클라이언트용 스마트 중계 프록시 스크립트 생성
proxy_code = '''import base64
import os
import tempfile
import requests
from fastapi import FastAPI, Request, Response
from fastapi.responses import JSONResponse
import uvicorn

app = FastAPI()
UPSTREAM_PORT = 9881

@app.get("/")
def root():
    return {"status": "ok", "message": "GPT-SoVITS Colab GPU Server Online!"}

@app.get("/tts")
def tts_get():
    return {"status": "ok", "service": "GPT-SoVITS TTS Engine"}

@app.post("/tts")
async def tts_proxy(request: Request):
    try:
        data = await request.json()
    except Exception:
        data = {}

    # 원격 클라이언트에서 base64로 전송된 참조 오디오가 있다면 임시 파일로 디코딩
    ref_b64 = data.get("ref_audio_base64")
    if ref_b64:
        try:
            audio_raw = base64.b64decode(ref_b64)
            tpath = os.path.join(tempfile.gettempdir(), "remote_ref_audio.wav")
            with open(tpath, "wb") as f:
                f.write(audio_raw)
            data["ref_audio_path"] = tpath
        except Exception as e:
            print(f"[Warning] Failed to decode base64 audio: {e}")

    # 내부 GPT-SoVITS 엔진으로 전달
    try:
        resp = requests.post(f"http://127.0.0.1:{UPSTREAM_PORT}/tts", json=data, timeout=120)
        content_type = resp.headers.get("content-type", "audio/wav")
        return Response(content=resp.content, status_code=resp.status_code, media_type=content_type)
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=9880, log_level="warning")
'''

with open("colab_proxy.py", "w", encoding="utf-8") as f:
    f.write(proxy_code)

print("🚀 [1/3] GPT-SoVITS 백그라운드 엔진 가동 중...")
api_cmd = [sys.executable, "api_v2.py", "-a", "127.0.0.1", "-p", "9881", "-c", "GPT_SoVITS/configs/tts_infer.yaml"]
api_log = open("/content/api_engine.log", "w", encoding="utf-8")
proc_api = subprocess.Popen(api_cmd, stdout=api_log, stderr=api_log)
time.sleep(5)

print("🚀 [2/3] 스마트 원격 프록시 서버 가동 중...")
proxy_cmd = [sys.executable, "colab_proxy.py"]
proxy_log = open("/content/proxy.log", "w", encoding="utf-8")
proc_proxy = subprocess.Popen(proxy_cmd, stdout=proxy_log, stderr=proxy_log)
time.sleep(3)

print("🌐 [3/3] Cloudflare 인터넷 공개 주소 생성 중...")
if os.path.exists("/content/tunnel.log"):
    try:
        os.remove("/content/tunnel.log")
    except Exception:
        pass

tunnel_cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:9880", "--logfile", "/content/tunnel.log"]
proc_tunnel = subprocess.Popen(tunnel_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

public_url = None
for _ in range(35):
    time.sleep(1)
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r", encoding="utf-8", errors="ignore") as f:
            log_data = f.read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_data)
            if m:
                public_url = m.group(0)
                break

if public_url:
    target_tts_url = f"{public_url}/tts"
    print("\n" + "="*72)
    print("🎉 [축하합니다! GPT-SoVITS 무료 GPU API 서버 준비 완료!]")
    print("="*72)
    print(f"\n👉 복사할 API 주소:\n   {target_tts_url}\n")
    print("="*72)
    print("📋 적용 방법:")
    print("1. 위 'https://...trycloudflare.com/tts' 주소를 마우스로 드래그하여 복사하세요.")
    print("2. AI Voice Studio 웹사이트 (https://voice-studio.streamlit.app/)로 이동합니다.")
    print("3. 좌측 사이드바 [GPT-SoVITS API 주소] 칸에 복사한 주소를 붙여넣습니다.")
    print("4. [🔌 서버 연결 테스트] 버튼을 눌러 초록색 '정상' 확인 후 사용하시면 됩니다!")
    print("="*72)
    print("\n💡 안내: 이 브라우저 탭을 끄지 않고 열어두시면 서버가 계속 유지됩니다.")
    
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("서버를 종료합니다.")
        proc_tunnel.terminate()
        proc_proxy.terminate()
        proc_api.terminate()
else:
    print("⚠️ 터널 주소를 생성하지 못했습니다. 셀을 다시 실행해주세요.")
